In [ ]:
# Clean environment
!pip uninstall -y transformers accelerate peft bitsandbytes

# Install known-good versions
!pip install -U transformers==4.45.2
!pip install -U accelerate peft datasets
!pip install -U bitsandbytes


Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
Found existing installation: accelerate 1.12.0
Uninstalling accelerate-1.12.0:
  Successfully uninstalled accelerate-1.12.0
Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 122.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 11.8 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.2/515.2 kB 48.3 MB/s eta

In [ ]:
import torch, transformers, bitsandbytes

print("Python OK")
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("bitsandbytes:", bitsandbytes.__version__)


Python OK
Transformers: 4.45.2
CUDA available: True
CUDA version: 12.6
bitsandbytes: 0.49.1


In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model

# =========================
# 1. Model & Tokenizer
# =========================
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
NEW_MODEL_NAME = "Qwen2.5-0.5B-Biology-Tutor"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# =========================
# 2. 4-bit Quantization
# =========================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# =========================
# 3. Dataset Loading
# =========================
dataset = load_dataset(
    "json",
    data_files="/content/biodata_clean.jsonl",
    split="train"
)

# =========================
# 4. Formatting + Tokenization
# =========================
def format_and_tokenize(example):
    messages = example["messages"]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    tokenized = tokenizer(
        text,
        truncation=True,
        max_length=1024,
        padding=False,
    )

    tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

tokenized_dataset = dataset.map(
    format_and_tokenize,
    remove_columns=dataset.column_names
)

# =========================
# 5. LoRA Configuration (Qwen-correct)
# =========================
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# =========================
# 6. Training Arguments
# =========================
training_args = TrainingArguments(
    output_dir="./bio_results",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    fp16=True,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none"
)

# =========================
# 7. Trainer
# =========================
trainer = Trainer(
    model=model,
    train_dataset=tokenized_dataset,
    args=training_args,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer,
        pad_to_multiple_of=8,
        return_tensors="pt"
    )
)

# =========================
# 8. Train
# =========================
trainer.train()

# =========================
# 9. Save LoRA Adapter + Tokenizer
# =========================
model.save_pretrained(NEW_MODEL_NAME)
tokenizer.save_pretrained(NEW_MODEL_NAME)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/425 [00:00<?, ? examples/s]

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


Step,Training Loss
10,2.076400
20,1.394900
30,1.186800
40,1.060700
50,0.964600
60,0.836000
70,0.756000


('Qwen2.5-0.5B-Biology-Tutor/tokenizer_config.json',
 'Qwen2.5-0.5B-Biology-Tutor/special_tokens_map.json',
 'Qwen2.5-0.5B-Biology-Tutor/vocab.json',
 'Qwen2.5-0.5B-Biology-Tutor/merges.txt',
 'Qwen2.5-0.5B-Biology-Tutor/added_tokens.json',
 'Qwen2.5-0.5B-Biology-Tutor/tokenizer.json')

In [ ]:
split = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
eval_ds = split["test"]


In [ ]:
from transformers import Trainer

eval_trainer = Trainer(
    model=model,
    args=training_args,
    eval_dataset=eval_ds,
    data_collator=DataCollatorForSeq2Seq(
        tokenizer,
        pad_to_multiple_of=8,
        return_tensors="pt"
    )
)


In [ ]:
import math

eval_results = eval_trainer.evaluate()
perplexity = math.exp(eval_results["eval_loss"])
print("Perplexity:", perplexity)


Perplexity: 1.9764110123523713


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = "Qwen2.5-0.5B-Biology-Tutor"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    trust_remote_code=True
)

model.eval()

# -------------------------
# Test prompt
# -------------------------
messages = [
    {
        "role": "system",
        "content": "You are a Kerala State Board Class 10 Biology teacher. Provide accurate, concise answers based on the Samagra Shiksha Keralam (SSLC) Biology syllabus."
    },
    {
        "role": "user",
        "content": "Name the structure in the eye that controls the amount of light entering the eye and state its function."
    }
]

# Convert messages → model prompt
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True   # 🔥 THIS IS CRITICAL
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# -------------------------
# Generate
# -------------------------
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=False,      # deterministic for exams
        temperature=0.0
    )

# Decode only generated text
generated_text = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print("MODEL ANSWER:\n", generated_text)


/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:601: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:606: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:623: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
Starting from v4.46, the `logits` model output will have the same type as the model (except at train 

MODEL ANSWER:
 The structure controlling the amount of light entering the eye is the Iris. It acts as a lens to adjust the refractive power of the cornea and lens, allowing different refractive powers for distant or near objects.


In [ ]:
from google.colab import drive
import shutil

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Copy the model folder to Drive
# This will create a folder named 'My-Bio-Model' in your Drive
destination = "/content/drive/MyDrive/My-Bio-Model"
shutil.copytree("Qwen2.5-0.5B-Biology-Tutor", destination)

print(f"Model saved permanently to: {destination}")

Mounted at /content/drive
Model saved permanently to: /content/drive/MyDrive/My-Bio-Model


In [ ]:
# ==========================================
# TEST THE TRAINED MODEL
# ==========================================

# 1. Prepare a test question
# We use the exact same System Prompt you trained with so the model knows its persona.
test_messages = [
    {"role": "system", "content": "You are a Kerala State Board Class 10 Biology teacher. Provide accurate, concise answers based on the Samagra Shiksha Keralam (SSLC) Biology syllabus."},
    {"role": "user", "content": "Explain the difference between Rod cells and Cone cells in the eye."}
]

# 2. Format and Tokenize
input_ids = tokenizer.apply_chat_template(
    test_messages,
    add_generation_prompt=True,
    return_tensors="pt"
).to("cuda")

# 3. Generate Answer
# We set temperature=0.3 for more factual/focused answers (good for science)
outputs = model.generate(
    input_ids,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.3,
    top_p=0.9
)

# 4. Decode and Print
response = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
print("\nTeacher's Answer:\n")
print(response)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



Teacher's Answer:

Rod cells: Detect low light levels (night vision). They contain blue-violet rods that are sensitive to weak light and colorless objects. Cone cells: Detect high light levels and colors. They contain red-yellow cones that are sensitive to bright light and vivid colors.


In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Load the Base Model
base_model_id = "Qwen/Qwen2.5-0.5B-Instruct"
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype=torch.float16
)

# 2. Load the Adapter (Your trained model)
# Make sure this points to the folder where you saved your model earlier
adapter_path = "Qwen2.5-0.5B-Biology-Tutor"
model = PeftModel.from_pretrained(base_model, adapter_path)

# 3. Merge
print("Merging model...")
merged_model = model.merge_and_unload()

# 4. Save the Merged Model
merged_output_dir = "Qwen-Bio-Merged"
merged_model.save_pretrained(merged_output_dir)
tokenizer = AutoTokenizer.from_pretrained(base_model_id)
tokenizer.save_pretrained(merged_output_dir)

print(f"✅ Success! Merged model saved to: {merged_output_dir}")

Merging model...
✅ Success! Merged model saved to: Qwen-Bio-Merged


In [ ]:
# 1. Clone llama.cpp
!git clone https://github.com/ggerganov/llama.cpp
%cd llama.cpp

# 2. Install requirements
!pip install -r requirements.txt

# 3. Convert to GGUF (FP16 - High Quality)
# This creates a 'qwen-bio-f16.gguf' file
!python convert_hf_to_gguf.py ../Qwen-Bio-Merged --outfile ../qwen-bio-f16.gguf --outtype f16

# 4. (Optional) Quantize to 4-bit (Smaller, Faster)
# This creates a 'qwen-bio-q4_k_m.gguf' file which is best for phones/laptops
!./llama-quantize ../qwen-bio-f16.gguf ../qwen-bio-q4_k_m.gguf q4_k_m

print("✅ GGUF files created: qwen-bio-f16.gguf and qwen-bio-q4_k_m.gguf")
%cd ..

Cloning into 'llama.cpp'...
remote: Enumerating objects: 77480, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 77480 (delta 34), reused 14 (delta 8), pack-reused 77407 (from 4)
Receiving objects: 100% (77480/77480), 282.88 MiB | 17.26 MiB/s, done.
Resolving deltas: 100% (56094/56094), done.
/content/llama.cpp
Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly, https://download.pytorch.org/whl/cpu, https://download.pytorch.org/whl/nightly
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
Ignoring torch: markers 'platform_machine == "s390x"' don't match your environment
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.6 MB/s eta 0:00:00
     ━━━━━━

INFO:hf-to-gguf:Loading model: Qwen-Bio-Merged
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {896, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {896}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> F16, shape = {4864, 896}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> F16, shape = {896, 4864}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> F16, shape = {896, 4864}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {896}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.float16 --> F32, shape = {128}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.float16 --> F16, shape = {896, 128}
INFO:hf-to-gguf:blk.0.attn_output.weight,  torch.float16 --> F1

In [ ]:
import os

# 1. Enter the directory
%cd /content/llama.cpp

# 2. COMPILE llama.cpp (This was missing!)
print("🔨 Building llama.cpp... (This takes 1-2 mins)")
!make -j

# 3. Check if the binary exists now
if os.path.exists("./llama-quantize"):
    print("✅ Build successful! Quantizing now...")

    # 4. Run Quantization again
    # Input: qwen-bio-f16.gguf (created in the previous step)
    # Output: qwen-bio-q4_k_m.gguf
    !./llama-quantize ../qwen-bio-f16.gguf ../qwen-bio-q4_k_m.gguf q4_k_m

    print("\n🎉 Conversion Complete!")
else:
    print("❌ Build failed. Please check the logs above.")

/content/llama.cpp
🔨 Building llama.cpp... (This takes 1-2 mins)
Makefile:6: *** Build system changed:
 The Makefile build has been replaced by CMake.

 For build instructions see:
 https://github.com/ggml-org/llama.cpp/blob/master/docs/build.md

.  Stop.
❌ Build failed. Please check the logs above.


In [ ]:
import os

# 1. Enter the directory
%cd /content/llama.cpp

# 2. Configure and Build using CMake
print("🔨 Building llama.cpp with CMake... (This takes 2-3 mins)")
!cmake -B build
!cmake --build build --config Release -j

print("✅ Build complete!")

/content/llama.cpp
🔨 Building llama.cpp with CMake... (This takes 2-3 mins)
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: 

In [ ]:
import os

# 1. Ensure we are in the llama.cpp folder
%cd /content/llama.cpp

# 2. Define paths
quantize_tool = "build/bin/llama-quantize"
input_model = "../qwen-bio-f16.gguf"   # The large file we made earlier
output_model = "../qwen-bio-q4_k_m.gguf" # The file we want

# 3. Check if tool exists
if os.path.exists(quantize_tool):
    print(f"✅ Found tool at {quantize_tool}. Starting quantization...")

    # 4. Run Quantization
    # usage: ./llama-quantize [input] [output] [method]
    !{quantize_tool} {input_model} {output_model} q4_k_m

    print("\n🎉 Conversion Complete! File is ready at:", output_model)

    # 5. Trigger Download
    from google.colab import files
    files.download(output_model)

else:
    print(f"❌ Could not find {quantize_tool}. Please check the build logs above.")

/content/llama.cpp
❌ Could not find build/bin/llama-quantize. Please check the build logs above.


In [ ]:
import os

# 1. Go to the folder
%cd /content/llama.cpp

# 2. Clean up the failed build attempt
!rm -rf build

# 3. Configure with CMake
print("⚙️ Configuring CMake...")
!cmake -B build

# 4. Build with LOW PARALLELISM to prevent crashing
# We use '-j 2' to limit it to 2 concurrent jobs.
print("🔨 Building llama.cpp (Slow & Safe Mode)...")
!cmake --build build --config Release -j 2

# 5. Check if it worked this time
quantize_path = "build/bin/llama-quantize"
if os.path.exists(quantize_path):
    print("\n✅ Build Success! The tool is ready.")

    # Run the conversion immediately
    input_model = "../qwen-bio-f16.gguf"
    output_model = "../qwen-bio-q4_k_m.gguf"

    print(f"🔄 Converting {input_model} to {output_model}...")
    !{quantize_path} {input_model} {output_model} q4_k_m

    print("🎉 Conversion Complete!")
else:
    print("❌ Build failed again. Try restarting the runtime (Runtime -> Restart Session).")

/content/llama.cpp
⚙️ Configuring CMake...
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-

In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

# Copy to Drive
shutil.copy("../qwen-bio-q4_k_m.gguf", "/content/drive/MyDrive/qwen-bio-q4_k_m.gguf")
print("✅ Saved to Google Drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


FileNotFoundError: [Errno 2] No such file or directory: '../qwen-bio-q4_k_m.gguf'

In [ ]:
import os
from google.colab import drive
import shutil

# 1. Setup paths
base_path = "/content/llama.cpp"
merged_model_path = "/content/Qwen-Bio-Merged"
f16_output = "/content/qwen-bio-f16.gguf"
q4_output = "/content/qwen-bio-q4_k_m.gguf"

# 2. Check if your merged model still exists
if not os.path.exists(merged_model_path):
    print("❌ ERROR: The folder 'Qwen-Bio-Merged' is missing!")
    print("This happens if you restarted the runtime. You must scroll up and run the 'Step 1: Merge the Model' code block again first.")
else:
    print("✅ Found merged model. Starting conversion...")

    # 3. Install Python requirements for the conversion script
    print("📦 Installing requirements...")
    !pip install -q -r {base_path}/requirements.txt

    # 4. Convert to F16 (Python Script)
    print("🔄 Step 1: Converting to F16 GGUF...")
    !python {base_path}/convert_hf_to_gguf.py {merged_model_path} --outfile {f16_output} --outtype f16

    # 5. Quantize to 4-bit (C++ Tool)
    print("mb Step 2: Quantizing to 4-bit...")
    quantize_tool = f"{base_path}/build/bin/llama-quantize"
    !{quantize_tool} {f16_output} {q4_output} q4_k_m

    # 6. Save to Google Drive
    print("💾 Step 3: Saving to Google Drive...")
    drive.mount('/content/drive')
    drive_dest = "/content/drive/MyDrive/qwen-bio-q4_k_m.gguf"
    shutil.copy(q4_output, drive_dest)

    print(f"\n🎉 SUCCESS! Your model is saved at: {drive_dest}")
    print("You can now download it from your Google Drive and use it in MLC Chat or LM Studio.")

✅ Found merged model. Starting conversion...
📦 Installing requirements...
🔄 Step 1: Converting to F16 GGUF...
INFO:hf-to-gguf:Loading model: Qwen-Bio-Merged
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {896, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {896}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> F16, shape = {4864, 896}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> F16, shape = {896, 4864}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> F16, shape = {896, 4864}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {896}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.float16 --> F32, shape = {128}
INFO:hf-to-gguf:blk.0.attn_k.weight,  

In [ ]:
import os
import shutil
from google.colab import drive

# ===============================================
# 1. CORRECT INSTALLATION
# ===============================================
print("📦 Installing MLC LLM (Universal Version)...")
# We remove the specific 'cu123' suffix so pip finds the best match
!pip install --pre -U -f https://mlc.ai/wheels mlc-llm-nightly mlc-ai-nightly

# ===============================================
# 2. DEFINE PATHS
# ===============================================
# Make sure your merged model folder still exists!
merged_model_path = "/content/Qwen-Bio-Merged"
mlc_output_path = "/content/Qwen-Bio-MLC"

if not os.path.exists(merged_model_path):
    print("❌ ERROR: 'Qwen-Bio-Merged' folder is missing.")
    print("Please scroll up and re-run 'Step 1: Merge the Model' first.")
else:
    # ===============================================
    # 3. CONVERT WEIGHTS
    # ===============================================
    print(f"🔄 Converting weights to MLC format...")
    # q4f16_1 is the standard compression for mobile phones
    !mlc_llm convert_weight {merged_model_path} \
        --quantization q4f16_1 \
        -o {mlc_output_path}

    # ===============================================
    # 4. GENERATE CONFIG
    # ===============================================
    print("⚙️ Generating MLC Config...")
    # Qwen2.5 uses the 'qwen2' template
    !mlc_llm gen_config {merged_model_path} \
        --quantization q4f16_1 \
        --conv-template qwen2 \
        -o {mlc_output_path}

    print("\n✅ MLC Conversion Complete!")

    # ===============================================
    # 5. ZIP AND SAVE
    # ===============================================
    print("💾 Zipping and saving to Google Drive...")

    # Create zip file
    shutil.make_archive("/content/qwen-bio-mlc", 'zip', mlc_output_path)

    # Move to Drive
    drive.mount('/content/drive')
    target_path = "/content/drive/MyDrive/qwen-bio-mlc.zip"
    shutil.copy("/content/qwen-bio-mlc.zip", target_path)

    print(f"🎉 DONE! The file is saved at: {target_path}")
    print("You can download this zip, unzip it, and copy the folder to your phone.")

📦 Installing MLC LLM (Universal Version)...
Looking in links: https://mlc.ai/wheels
ERROR: Could not find a version that satisfies the requirement mlc-llm-nightly (from versions: none)
ERROR: No matching distribution found for mlc-llm-nightly
🔄 Converting weights to MLC format...
/bin/bash: line 1: mlc_llm: command not found
⚙️ Generating MLC Config...
/bin/bash: line 1: mlc_llm: command not found

✅ MLC Conversion Complete!
💾 Zipping and saving to Google Drive...


FileNotFoundError: [Errno 2] No such file or directory: '/content/Qwen-Bio-MLC'